# 🧠 Fine-Tune Qwen2-VL-2B ด้วย 4-bit QLoRA สำหรับอ่านโพยกระดาษภาษาไทย

สมุดโค้ดนี้ออกแบบมาเพื่อปรับแต่งโมเดล Vision-Language ขนาดกะทัดรัด **Qwen2-VL-2B-Instruct** ให้จดจำและถอดข้อความจากภาพถ่ายโพยกระดาษภาษาไทยได้อย่างแม่นยำสูง

### ⚡️ คุณสมบัติเด่น:
- **ฟรี 100% บน Google Colab T4 GPU**: ใช้ VRAM เพียง ~6-7 GB จากโควต้า 15-16 GB
- **4-bit BitsAndBytes NF4 Quantization**: โหลดโมเดลตัวเบาแต่ยังรักษาคุณภาพระดับสูง
- **LoRA (Low-Rank Adaptation)**: ปรับแต่งน้ำหนักเฉพาะส่วน ไม่เปลืองหน่วยความจำ และเทรนเสร็จใน 10-15 นาที
- **ส่งออกไฟล์ LoRA Adapter ทันที**: ได้ไฟล์ zip เพื่อนํากลับมาใช้แทนหรือเสริมโมเดลหลักในระบบ

In [ ]:
# 1. ตรวจสอบการเชื่อมต่อ GPU T4
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ กรุณาเปลี่ยน Runtime เป็น T4 GPU โดยไปที่ Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# 2. ติดตั้งไลบรารีที่จำเป็นสำหรับ VLM QLoRA
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q accelerate peft bitsandbytes datasets torchvision pillow qwen-vl-utils

In [ ]:
# 3. แตกไฟล์ชุดข้อมูล Paper OCR Training Bundle
import os, zipfile, json

# ค้นหาไฟล์ zip ในไดเรกทอรีปัจจุบัน
zip_files = [f for f in os.listdir(".") if f.endswith(".zip") and "bundle" in f.lower()]
if not zip_files and os.path.exists("paper_ocr_training_bundle.zip"):
    zip_files = ["paper_ocr_training_bundle.zip"]

if zip_files:
    target_zip = zip_files[0]
    print(f"📦 แตกไฟล์: {target_zip}")
    with zipfile.ZipFile(target_zip, "r") as z:
        z.extractall("./data")
    print("✅ แตกไฟล์ข้อมูลสำเร็จ!")
elif os.path.exists("dataset.jsonl"):
    os.makedirs("./data", exist_ok=True)
    !cp dataset.jsonl ./data/
    if os.path.exists("images"):
        !cp -r images ./data/
    print("✅ พบไฟล์ dataset.jsonl ในโฟลเดอร์หลัก")
else:
    print("⚠️ ไม่พบไฟล์ bundle zip กรุณาลากไฟล์ zip จากเครื่องมาวางในหน้าต่าง Files ด้านซ้ายของ Colab")

In [ ]:
# 4. ตรวจสอบข้อมูลตัวอย่างและรูปภาพโพย
from PIL import Image

data_file = "./data/dataset.jsonl"
if os.path.exists(data_file):
    with open(data_file, "r", encoding="utf-8") as f:
        samples = [json.loads(line) for line in f if line.strip()]
    print(f"📊 จำนวนตัวอย่างสำหรับเทรน: {len(samples)} ใบ")
    if samples:
        s0 = samples[0]
        print(f"ใบที่: {s0.get('sheet_id', '-')}")
        print("คำสั่ง (Prompt):", s0["conversations"][0]["value"])
        print("เฉลย (Ground Truth):
", s0["conversations"][1]["value"][:200], "...")
        img_full = os.path.join("./data", s0["image"])
        if os.path.exists(img_full):
            img = Image.open(img_full)
            w, h = img.size
            display(img.resize((min(w, 400), int(h * min(w, 400) / w))))
else:
    print("ไม่พบ ./data/dataset.jsonl")

In [ ]:
# 5. โหลด Qwen2-VL-2B-Instruct พร้อมตั้งค่า 4-bit Quantization
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model_id = "Qwen/Qwen2-VL-2B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("⏳ กำลังดาวน์โหลดโมเดลและโปรเซสเซอร์...")
processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=1024*28*28)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(model, lora_config)
print("
🔥 พารามิเตอร์ที่เปิดให้ LoRA ปรับแต่ง:")
peft_model.print_trainable_parameters()

In [ ]:
# 6. เตรียมโครงสร้าง Dataset สำหรับการเทรน
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

class PaperOCRDataset(Dataset):
    def __init__(self, samples, data_root="./data"):
        self.samples = samples
        self.data_root = data_root

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        img_full = os.path.join(self.data_root, item["image"])
        image = Image.open(img_full).convert("RGB")
        
        query = item["conversations"][0]["value"].replace("<image>\n", "").replace("<image>", "")
        response = item["conversations"][1]["value"]
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": query}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": response}
                ]
            }
        ]
        
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )
        item_dict = {k: v.squeeze(0) for k, v in inputs.items()}
        item_dict["labels"] = item_dict["input_ids"].clone()
        return item_dict

train_dataset = PaperOCRDataset(samples)
print(f"สร้าง Dataset สำเร็จ! จำนวน {len(train_dataset)} ตัวอย่าง")

In [ ]:
# 7. เริ่มต้นการเทรน (Training Loop)
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./output_qwen2_vl_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    num_train_epochs=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    report_to="none"
)

def collate_fn(batch):
    keys = batch[0].keys()
    collated = {}
    for k in keys:
        items = [b[k] for b in batch]
        if k in ["input_ids", "labels", "attention_mask"]:
            padded = torch.nn.utils.rnn.pad_sequence(
                items, batch_first=True, padding_value=processor.tokenizer.pad_token_id or 0
            )
            collated[k] = padded
        elif k in ["pixel_values", "image_grid_thw"]:
            collated[k] = torch.cat(items, dim=0)
        else:
            collated[k] = items
    return collated

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

print("🚀 กำลังเริ่มกระบวนการ Fine-Tuning บน Colab T4 GPU...")
trainer.train()
print("🎉 การฝึกฝนเสร็จสมบูรณ์เรียบร้อย!")

In [ ]:
# 8. บันทึก LoRA Adapter และแพ็คเป็น ZIP ดาวน์โหลดกลับมาใช้งาน
adapter_dir = "./paper_ocr_lora_adapter"
peft_model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
print(f"✅ บันทึก LoRA Adapter ไว้ที่ {adapter_dir}")

!zip -r paper_ocr_lora_adapter.zip ./paper_ocr_lora_adapter
print("📦 บีบอัดไฟล์เป็น paper_ocr_lora_adapter.zip สำเร็จ!")

from google.colab import files
files.download("paper_ocr_lora_adapter.zip")